#  Kuli AI — Qwen2.5-Coder-7B Fine-Tuning with Unsloth

**Optimizations in this notebook:**
-  **Flash Attention 2** — 3-4x faster training
-  **Sequence Packing** — eliminates padding waste, 2x throughput
-  **Gradient Checkpointing (Unsloth)** — 40% VRAM reduction
-  **8-bit AdamW** — memory-efficient optimizer
-  **Live eval loss tracking** on held-out `eval.jsonl`
-  **Auto GGUF export** → drag into Ollama with one command

**GPU Requirement:** Free Google Colab T4 (16GB VRAM) — fits comfortably with 4-bit QLoRA.

In [ ]:
# ── Step 1: Install Unsloth and dependencies ──────────────────────────────────
import sys
import os
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps xformers trl peft accelerate bitsandbytes -q

In [ ]:
# ── Step 2: Load base model with Flash Attention 2 ───────────────────────────
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Qwen2.5-Coder supports up to 32k, 2048 is ideal for our data
dtype = None           # Auto-detect (bfloat16 on A100, float16 on T4)
load_in_4bit = True    # 4-bit QLoRA: reduces 7B from ~14GB to ~4GB VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # Flash Attention 2 is auto-enabled by Unsloth — 3-4x faster than standard attention
)
print(f"Model loaded. Parameters: {model.num_parameters():,}")

In [ ]:
# ── Step 3: Attach LoRA adapters ─────────────────────────────────────────────
# We inject LoRA into ALL projection layers for maximum adaptation
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                    # Rank: controls adapter expressiveness
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention projections
        "gate_proj", "up_proj", "down_proj",       # MLP layers
    ],
    lora_alpha = 16,           # Scaling factor (same as r = balanced)
    lora_dropout = 0,          # 0 is optimal for Unsloth (no dropout needed)
    bias = "none",             # Don't train bias terms
    use_gradient_checkpointing = "unsloth",  # 40% VRAM reduction
    random_state = 42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100*trainable/total:.2f}% of {total:,} total)")

In [ ]:
# ── Step 4: Load datasets ─────────────────────────────────────────────────────
# Load dataset files (train.jsonl, eval.jsonl)
from datasets import load_dataset

train_dataset = load_dataset('json', data_files='/content/train.jsonl', split='train')
eval_dataset  = load_dataset('json', data_files='/content/eval.jsonl',  split='train')

print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples:  {len(eval_dataset)}")
print(f"\nSample entry preview:")
print(train_dataset[0]['text'][:300] + '...')

In [ ]:
# ── Step 5: Configure SFT Trainer ────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,        #  Sequence packing: batches multiple examples per sequence → 2x throughput
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        per_device_eval_batch_size  = 2,
        gradient_accumulation_steps = 4,    # Effective batch = 8
        warmup_ratio = 0.05,
        num_train_epochs = 3,               # 3 full passes over the 500 training examples
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = 20,                    # Evaluate every 20 steps
        save_strategy = "steps",
        save_steps = 20,
        load_best_model_at_end = True,      # Auto-save the checkpoint with lowest eval loss
        metric_for_best_model = "eval_loss",
        optim = "adamw_8bit",               #  8-bit optimizer: 4x less memory than fp32 Adam
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",       # Cosine decay for smoother convergence
        seed = 42,
        output_dir = "kuli-qwen-7b-checkpoints",
        report_to = "none",
    ),
)
print("Trainer configured. Ready to train.")

In [ ]:
# ── Step 6: Train! ────────────────────────────────────────────────────────────
print("Starting Unsloth QLoRA fine-tuning...")
trainer_stats = trainer.train()
print(f"\n Training complete!")
print(f"   Final train loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# ── Step 7: Evaluate final model on held-out eval set ────────────────────────
eval_results = trainer.evaluate()
print(f"\n📊 Evaluation Results:")
for k, v in eval_results.items():
    print(f"   {k}: {v:.4f}")

In [ ]:
# ── Step 8: Export to GGUF for Ollama ────────────────────────────────────────
# This merges the LoRA adapters into the 7B base weights and quantizes to Q4_K_M.
# Q4_K_M is the industry standard: good quality, ~4.1 GB file size.
print("Exporting merged model to GGUF (Q4_K_M quantization)...")
model.save_pretrained_gguf(
    "kuli-qwen-7b",
    tokenizer,
    quantization_method = "q4_k_m"
)
print("\n GGUF export complete!")
print("   File: kuli-qwen-7b-unsloth.Q4_K_M.gguf")
print("\nNext steps (run on your Mac):")
print("  1. Download the .gguf file from Kaggle/Colab")
print("  2. Create a Modelfile:")
print('     echo "FROM ./kuli-qwen-7b-unsloth.Q4_K_M.gguf" > Modelfile')
print("  3. Register in Ollama:")
print("     ollama create kuli-qwen-7b -f Modelfile")
print("  4. Select 'kuli-qwen-7b' in the Kuli AI sidebar!")